# EDA Test Runner

Exercises every function from **EDA_Governance.ipynb** and **EDA_Correlation.ipynb** against `raw_test_call_data.csv`.

In [ ]:
# --- Setup: imports and theme ---
import sys, os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, os.path.abspath('../../src'))
from elocal_analysis.elocal_theme import (
    set_elocal_theme, ELOCAL_PALETTE, ELOCAL_BLUE, ELOCAL_ORANGE, ELOCAL_GREEN,
    ELOCAL_SEQ_CMAP
)
set_elocal_theme()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
print('Setup complete.')

In [ ]:
# --- Load the test dataset ---
DATA_PATH = os.path.join('../../data/raw/', 'raw_test_call_data.csv')
df = pd.read_csv(DATA_PATH)
print(f'Loaded {DATA_PATH}')
print(f'Shape: {df.shape}')
df.head()

---
# Part 1 — EDA_Governance Functions

## Completeness

In [ ]:
# --- Test: completeness_summary ---
def completeness_summary(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    missing_count = df.isnull().sum()
    result = pd.DataFrame({
        'dtype': df.dtypes,
        'missing_count': missing_count,
        'missing_pct': (missing_count / n * 100).round(2),
        'present_pct': ((1 - missing_count / n) * 100).round(2)
    }).sort_values('missing_pct', ascending=False)
    return result

completeness_summary(df)

In [ ]:
# --- Test: plot_missing ---
def plot_missing(df: pd.DataFrame, threshold: float = 0.0) -> None:
    pct = (df.isnull().mean() * 100).sort_values(ascending=False)
    pct = pct[pct > threshold]
    if pct.empty:
        print('No columns above the missing-value threshold.')
        return
    fig, ax = plt.subplots(figsize=(16, max(4, len(pct) * 0.4)))
    sns.barplot(x=pct.values, y=pct.index, color=ELOCAL_BLUE, ax=ax)
    ax.set_xlabel('% Missing')
    ax.set_title('Missing Values by Column')
    for i, v in enumerate(pct.values):
        ax.text(v + 0.3, i, f'{v:.1f}%', va='center')
    plt.tight_layout()
    plt.show()

plot_missing(df)

In [ ]:
# --- Test: plot_missingness_heatmap ---
def plot_missingness_heatmap(df: pd.DataFrame) -> None:
    sample = df.sample(min(500, len(df)), random_state=42) if len(df) > 500 else df
    fig, ax = plt.subplots(figsize=(16, 8))
    sns.heatmap(sample.isnull().astype(int), cbar=False,
                yticklabels=False, cmap=[ELOCAL_GREEN, ELOCAL_ORANGE], ax=ax)
    ax.set_title('Missingness Pattern (green = present, orange = missing)')
    plt.tight_layout()
    plt.show()

plot_missingness_heatmap(df)

In [ ]:
# --- Test: missingness_diagnostic ---
def missingness_diagnostic(df: pd.DataFrame) -> pd.DataFrame:
    cols_with_missing = [c for c in df.columns if df[c].isnull().any()]
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    rows = []
    for col in cols_with_missing:
        mask = df[col].isnull()
        n_miss = mask.sum()
        pct = (n_miss / len(df) * 100)
        sig_cols = []
        for nc in numeric_cols:
            if nc == col:
                continue
            grp_present = df.loc[~mask, nc].dropna()
            grp_missing = df.loc[mask, nc].dropna()
            if len(grp_present) < 2 or len(grp_missing) < 2:
                continue
            _, p = stats.ttest_ind(grp_present, grp_missing, equal_var=False)
            if p < 0.05:
                sig_cols.append(nc)
        if len(sig_cols) == 0:
            pattern = 'Likely MCAR'
        else:
            pattern = f'Likely MAR (differs on: {", ".join(sig_cols[:5])})'
        rows.append({'column': col, 'missing_n': n_miss,
                      'missing_pct': round(pct, 2), 'pattern': pattern})
    return pd.DataFrame(rows)

missingness_diagnostic(df)

## Accuracy

In [ ]:
# --- Test: iqr_outliers (single column) ---
def iqr_outliers(df: pd.DataFrame, col: str, factor: float = 1.5) -> pd.DataFrame:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    result = df.loc[mask].copy()
    print(f"{col}: Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}  "
          f"bounds=[{lower:.2f}, {upper:.2f}]  outliers={mask.sum()} "
          f"({mask.mean()*100:.2f}%)")
    return result

iqr_outliers(df, 'call_duration')

In [ ]:
# --- Test: iqr_outlier_summary (all numeric) ---
def iqr_outlier_summary(df: pd.DataFrame, factor: float = 1.5) -> pd.DataFrame:
    numeric_cols = df.select_dtypes(include='number').columns
    rows = []
    for col in numeric_cols:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - factor * iqr, q3 + factor * iqr
        n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        rows.append({
            'column': col, 'Q1': q1, 'Q3': q3, 'IQR': iqr,
            'lower_bound': lower, 'upper_bound': upper,
            'outlier_count': n_outliers,
            'outlier_pct': round(n_outliers / len(df) * 100, 2)
        })
    return pd.DataFrame(rows).sort_values('outlier_pct', ascending=False)

iqr_outlier_summary(df)

In [ ]:
# --- Test: plot_outlier_boxplots ---
def plot_outlier_boxplots(df: pd.DataFrame) -> None:
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        print('No numeric columns.')
        return
    n_cols = min(3, len(numeric_cols))
    n_rows = -(-len(numeric_cols) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = np.array(axes).flatten()
    for i, col in enumerate(numeric_cols):
        sns.boxplot(y=df[col], ax=axes[i], color=ELOCAL_BLUE)
        axes[i].set_title(f'{col}')
    for j in range(len(numeric_cols), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Outlier Detection — Box Plots', fontsize=18, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

plot_outlier_boxplots(df)

In [ ]:
# --- Test: impossible_value_check ---
def impossible_value_check(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        issues = []
        if pd.api.types.is_numeric_dtype(df[col]):
            n_neg = (df[col] < 0).sum()
            if n_neg > 0:
                issues.append(f'{n_neg} negative values')
            n_zero = (df[col] == 0).sum()
            if n_zero > 0:
                issues.append(f'{n_zero} zeros')
        elif df[col].dtype == 'object':
            sample = df[col].dropna().head(1000)
            n_numeric_like = sample.apply(lambda x: str(x).replace('.', '', 1).replace('-', '', 1).isdigit()).sum()
            if 0 < n_numeric_like < len(sample):
                issues.append(f'mixed types ({n_numeric_like}/{len(sample)} look numeric)')
        if issues:
            rows.append({'column': col, 'dtype': str(df[col].dtype), 'issues': '; '.join(issues)})
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=['column', 'dtype', 'issues'])

impossible_value_check(df)

## Consistency

In [ ]:
# --- Test: duplicate_summary ---
def duplicate_summary(df: pd.DataFrame) -> dict:
    n_dup = df.duplicated().sum()
    result = {
        'total_rows': len(df),
        'duplicate_rows': n_dup,
        'duplicate_pct': round(n_dup / len(df) * 100, 2),
        'unique_rows': len(df) - n_dup
    }
    print(f"Duplicates: {n_dup} / {len(df)} ({result['duplicate_pct']}%)")
    return result

duplicate_summary(df)

In [ ]:
# --- Test: duplicate_on_keys ---
def duplicate_on_keys(df: pd.DataFrame, key_cols: list) -> pd.DataFrame:
    mask = df.duplicated(subset=key_cols, keep=False)
    dups = df.loc[mask].sort_values(key_cols)
    print(f"Rows duplicated on {key_cols}: {mask.sum()} / {len(df)}")
    return dups

duplicate_on_keys(df, ['disposition'])

In [ ]:
# --- Test: conflicting_entries ---
def conflicting_entries(df: pd.DataFrame, key_cols: list, value_col: str) -> pd.DataFrame:
    grouped = df.groupby(key_cols)[value_col].nunique().reset_index(name='n_unique')
    conflicts = grouped[grouped['n_unique'] > 1]
    print(f"Conflicting groups on {key_cols} for '{value_col}': {len(conflicts)}")
    return conflicts

conflicting_entries(df, ['disposition'], 'billable_status')

In [ ]:
# --- Test: referential_integrity ---
def referential_integrity(df: pd.DataFrame, fk_col: str,
                          ref_df: pd.DataFrame, ref_col: str) -> pd.DataFrame:
    ref_vals = set(ref_df[ref_col].dropna().unique())
    orphans = df[~df[fk_col].isin(ref_vals)]
    print(f"Orphan rows ({fk_col} not in reference): {len(orphans)} / {len(df)}")
    return orphans

# Create a small reference table to test against
valid_dispositions = pd.DataFrame({'disposition': ['Answered', 'Missed', 'Voicemail']})
referential_integrity(df, 'disposition', valid_dispositions, 'disposition')

## Cardinality

In [ ]:
# --- Test: cardinality_summary ---
def cardinality_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        n_unique = df[col].nunique(dropna=True)
        rows.append({
            'column': col,
            'dtype': str(df[col].dtype),
            'n_unique': n_unique,
            'cardinality_ratio': round(n_unique / len(df) * 100, 2)
        })
    return pd.DataFrame(rows).sort_values('n_unique', ascending=False)

cardinality_summary(df)

In [ ]:
# --- Test: value_counts_pct ---
def value_counts_pct(df: pd.DataFrame, col: str, top_n: int = 20) -> pd.DataFrame:
    vc = df[col].value_counts(dropna=False).head(top_n).reset_index()
    vc.columns = ['value', 'count']
    vc['pct'] = (vc['count'] / len(df) * 100).round(2)
    return vc

value_counts_pct(df, 'disposition')

In [ ]:
# --- Test: plot_value_counts ---
def plot_value_counts(df: pd.DataFrame, col: str, top_n: int = 20) -> None:
    vc = df[col].value_counts(dropna=False).head(top_n)
    fig, ax = plt.subplots(figsize=(16, max(4, len(vc) * 0.4)))
    sns.barplot(x=vc.values, y=vc.index.astype(str), color=ELOCAL_ORANGE, ax=ax)
    ax.set_xlabel('Count')
    ax.set_title(f'Top {top_n} Values — {col}')
    for i, v in enumerate(vc.values):
        ax.text(v + 0.3, i, f'{v}', va='center')
    plt.tight_layout()
    plt.show()

plot_value_counts(df, 'disposition')

## Univariate Analysis

In [ ]:
# --- Test: univariate_stats ---
def univariate_stats(df: pd.DataFrame) -> pd.DataFrame:
    desc = df.describe().T
    desc['skew'] = df.select_dtypes(include='number').skew()
    desc['kurtosis'] = df.select_dtypes(include='number').kurtosis()
    desc['iqr'] = desc['75%'] - desc['25%']
    desc['cv'] = (desc['std'] / desc['mean']).round(4)
    return desc

univariate_stats(df)

In [ ]:
# --- Test: plot_distributions ---
def plot_distributions(df: pd.DataFrame) -> None:
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        print('No numeric columns.')
        return
    n_cols = min(3, len(numeric_cols))
    n_rows = -(-len(numeric_cols) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = np.array(axes).flatten()
    for i, col in enumerate(numeric_cols):
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i],
                     color=ELOCAL_BLUE, edgecolor='white')
        skew_val = df[col].skew()
        axes[i].set_title(f'{col}  (skew={skew_val:.2f})')
    for j in range(len(numeric_cols), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Distributions — Numeric Columns', fontsize=18,
                 fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

plot_distributions(df)

In [ ]:
# --- Test: plot_single_distribution ---
def plot_single_distribution(df: pd.DataFrame, col: str) -> None:
    data = df[col].dropna()
    fig, ax = plt.subplots(figsize=(16, 6))
    sns.histplot(data, kde=True, color=ELOCAL_BLUE, edgecolor='white', ax=ax)
    ax.axvline(data.mean(), color=ELOCAL_ORANGE, linestyle='--', label=f'Mean: {data.mean():.2f}')
    ax.axvline(data.median(), color=ELOCAL_GREEN, linestyle='-', label=f'Median: {data.median():.2f}')
    ax.legend()
    ax.set_title(f'Distribution of {col} (skew={data.skew():.2f}, kurtosis={data.kurtosis():.2f})')
    plt.tight_layout()
    plt.show()

plot_single_distribution(df, 'call_duration')

## Bivariate (Governance quick versions)

In [ ]:
# --- Test: top_correlations (governance version) ---
def top_correlations_gov(df: pd.DataFrame, n: int = 15) -> pd.DataFrame:
    corr = df.select_dtypes(include='number').corr()
    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs.columns = ['feature_1', 'feature_2', 'correlation']
    pairs['abs_corr'] = pairs['correlation'].abs()
    return pairs.sort_values('abs_corr', ascending=False).head(n).drop(columns='abs_corr')

top_correlations_gov(df)

In [ ]:
# --- Test: plot_correlation_heatmap (governance version) ---
def plot_correlation_heatmap_gov(df: pd.DataFrame) -> None:
    numeric_df = df.select_dtypes(include='number')
    if numeric_df.shape[1] < 2:
        print('Need at least 2 numeric columns.')
        return
    corr = numeric_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    fig, ax = plt.subplots(figsize=(16, 12))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                cmap='coolwarm', center=0, square=True,
                linewidths=0.5, ax=ax)
    ax.set_title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()

plot_correlation_heatmap_gov(df)

In [ ]:
# --- Test: target_leakage_check (governance version) ---
def target_leakage_check_gov(df: pd.DataFrame, target_col: str,
                              threshold: float = 0.95) -> pd.DataFrame:
    numeric_df = df.select_dtypes(include='number')
    if target_col not in numeric_df.columns:
        print(f'{target_col} is not numeric — skipping leakage check.')
        return pd.DataFrame()
    corr_with_target = numeric_df.corr()[target_col].drop(target_col).abs()
    leakers = corr_with_target[corr_with_target >= threshold]
    if leakers.empty:
        print(f'No features above {threshold} correlation with {target_col}.')
    else:
        print(f'POTENTIAL LEAKAGE — features with |corr| >= {threshold}:')
        print(leakers.sort_values(ascending=False))
    return leakers.reset_index().rename(columns={'index': 'feature', target_col: 'abs_corr'})

target_leakage_check_gov(df, 'call_duration')

---
# Part 2 — EDA_Correlation Functions

## Correlation Matrix

In [ ]:
# --- Test: correlation_matrix ---
def correlation_matrix(df: pd.DataFrame, method: str = 'pearson') -> pd.DataFrame:
    corr = df.select_dtypes(include='number').corr(method=method)
    return corr

print('Pearson:')
display(correlation_matrix(df, 'pearson'))
print('\nSpearman:')
display(correlation_matrix(df, 'spearman'))

In [ ]:
# --- Test: top_correlations (correlation version, with method param) ---
def top_correlations(df: pd.DataFrame, n: int = 20,
                     method: str = 'pearson') -> pd.DataFrame:
    corr = df.select_dtypes(include='number').corr(method=method)
    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs.columns = ['feature_1', 'feature_2', 'correlation']
    pairs['abs_corr'] = pairs['correlation'].abs()
    return pairs.sort_values('abs_corr', ascending=False).head(n).drop(columns='abs_corr')

top_correlations(df)

## Correlation Heatmap

In [ ]:
# --- Test: plot_correlation_heatmap (full version with method param) ---
def plot_correlation_heatmap(df: pd.DataFrame, method: str = 'pearson',
                              figsize: tuple = (16, 12)) -> None:
    numeric_df = df.select_dtypes(include='number')
    if numeric_df.shape[1] < 2:
        print('Need at least 2 numeric columns for a heatmap.')
        return
    corr = numeric_df.corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                cmap='coolwarm', center=0, square=True,
                linewidths=0.5, ax=ax,
                cbar_kws={'shrink': 0.8, 'label': f'{method.title()} r'})
    ax.set_title(f'{method.title()} Correlation Heatmap', pad=20)
    plt.tight_layout()
    plt.show()

plot_correlation_heatmap(df, method='spearman')

In [ ]:
# --- Test: plot_clustered_heatmap ---
def plot_clustered_heatmap(df: pd.DataFrame, method: str = 'pearson') -> None:
    numeric_df = df.select_dtypes(include='number')
    if numeric_df.shape[1] < 2:
        print('Need at least 2 numeric columns.')
        return
    corr = numeric_df.corr(method=method)
    g = sns.clustermap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                       center=0, linewidths=0.5, figsize=(14, 12),
                       cbar_kws={'label': f'{method.title()} r'})
    g.fig.suptitle(f'Clustered {method.title()} Correlation', y=1.02,
                   fontsize=18, fontweight='bold')
    plt.show()

plot_clustered_heatmap(df)

## Scatter & Pair Plots

In [ ]:
# --- Test: plot_scatter ---
def plot_scatter(df: pd.DataFrame, x: str, y: str,
                 hue: str = None) -> None:
    fig, ax = plt.subplots(figsize=(16, 9))
    sns.scatterplot(data=df, x=x, y=y, hue=hue,
                    palette=ELOCAL_PALETTE, alpha=0.6, ax=ax)
    sns.regplot(data=df, x=x, y=y, scatter=False,
                color=ELOCAL_ORANGE, ax=ax, line_kws={'linewidth': 2})
    r, p = stats.pearsonr(df[x].dropna(), df[y].dropna())
    ax.set_title(f'{y} vs {x}  (r={r:.3f}, p={p:.3e})')
    plt.tight_layout()
    plt.show()

plot_scatter(df, 'call_id', 'call_duration')

In [ ]:
# --- Test: plot_pairplot ---
def plot_pairplot(df: pd.DataFrame, cols: list = None,
                  hue: str = None) -> None:
    if cols is None:
        cols = df.select_dtypes(include='number').columns.tolist()
    if len(cols) > 8:
        print(f'{len(cols)} columns — using first 8 to keep plot readable.')
        cols = cols[:8]
    subset = cols + ([hue] if hue and hue not in cols else [])
    g = sns.pairplot(df[subset].dropna(), hue=hue,
                     palette=ELOCAL_PALETTE, diag_kind='kde',
                     plot_kws={'alpha': 0.5})
    g.fig.suptitle('Pair Plot', y=1.02, fontsize=18, fontweight='bold')
    plt.show()

plot_pairplot(df, ['call_id', 'call_duration', 'zip_code'])

## Categorical Cross-Tabs

In [ ]:
# --- Test: crosstab_analysis ---
def crosstab_analysis(df: pd.DataFrame, col_a: str, col_b: str,
                      normalize: str = 'index') -> pd.DataFrame:
    ct = pd.crosstab(df[col_a], df[col_b], normalize=normalize)
    return ct

crosstab_analysis(df, 'disposition', 'billable_status')

In [ ]:
# --- Test: plot_crosstab_heatmap ---
def plot_crosstab_heatmap(df: pd.DataFrame, col_a: str, col_b: str) -> None:
    ct = pd.crosstab(df[col_a], df[col_b])
    fig, ax = plt.subplots(figsize=(16, 9))
    sns.heatmap(ct, annot=True, fmt='d', cmap=ELOCAL_SEQ_CMAP,
                linewidths=0.5, ax=ax)
    ax.set_title(f'Cross-Tab: {col_a} x {col_b}')
    plt.tight_layout()
    plt.show()

plot_crosstab_heatmap(df, 'disposition', 'billable_status')

In [ ]:
# --- Test: chi_squared_test ---
def chi_squared_test(df: pd.DataFrame, col_a: str, col_b: str) -> dict:
    ct = pd.crosstab(df[col_a], df[col_b])
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    result = {'chi2': round(chi2, 4), 'p_value': p, 'dof': dof,
              'significant_at_05': p < 0.05}
    print(f"Chi2 = {chi2:.4f}, p = {p:.3e}, dof = {dof}  ->  "
          f"{'SIGNIFICANT' if p < 0.05 else 'not significant'} at a=0.05")
    return result

chi_squared_test(df, 'disposition', 'billable_status')

## Target Leakage Detection

In [ ]:
# --- Test: target_leakage_check (correlation version) ---
def target_leakage_check(df: pd.DataFrame, target_col: str,
                          threshold: float = 0.95) -> pd.DataFrame:
    numeric_df = df.select_dtypes(include='number')
    if target_col not in numeric_df.columns:
        print(f'{target_col} is not numeric — cannot compute correlation.')
        return pd.DataFrame()
    corr = numeric_df.corr()[target_col].drop(target_col).abs().sort_values(ascending=False)
    leakers = corr[corr >= threshold]
    if leakers.empty:
        print(f'No features above {threshold} correlation with "{target_col}".')
    else:
        print(f'POTENTIAL LEAKAGE — features with |corr| >= {threshold} to "{target_col}":')
    return leakers.reset_index().rename(columns={'index': 'feature', target_col: 'abs_corr'})

target_leakage_check(df, 'call_duration')

In [ ]:
# --- Test: plot_target_correlations ---
def plot_target_correlations(df: pd.DataFrame, target_col: str,
                              top_n: int = 20) -> None:
    numeric_df = df.select_dtypes(include='number')
    if target_col not in numeric_df.columns:
        print(f'{target_col} is not numeric.')
        return
    corr = numeric_df.corr()[target_col].drop(target_col).sort_values(key=abs, ascending=False).head(top_n)
    colors = [ELOCAL_ORANGE if abs(v) >= 0.95 else ELOCAL_BLUE for v in corr.values]
    fig, ax = plt.subplots(figsize=(16, max(4, len(corr) * 0.4)))
    sns.barplot(x=corr.values, y=corr.index, palette=colors, ax=ax)
    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set_xlabel('Correlation')
    ax.set_title(f'Top {top_n} Feature Correlations with "{target_col}"\n(orange = potential leakage >= 0.95)')
    plt.tight_layout()
    plt.show()

plot_target_correlations(df, 'call_duration')

## Numeric vs Categorical Relationships

In [ ]:
# --- Test: plot_numeric_by_category ---
def plot_numeric_by_category(df: pd.DataFrame, numeric_col: str,
                              cat_col: str, top_n: int = 10) -> None:
    top_cats = df[cat_col].value_counts().head(top_n).index
    subset = df[df[cat_col].isin(top_cats)]
    fig, ax = plt.subplots(figsize=(16, 8))
    sns.boxplot(data=subset, x=cat_col, y=numeric_col,
                palette=ELOCAL_PALETTE, ax=ax)
    ax.set_title(f'{numeric_col} by {cat_col} (top {top_n} categories)')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

plot_numeric_by_category(df, 'call_duration', 'disposition')

In [ ]:
# --- Test: anova_test ---
def anova_test(df: pd.DataFrame, numeric_col: str,
               cat_col: str) -> dict:
    groups = [g[numeric_col].dropna().values
              for _, g in df.groupby(cat_col) if len(g[numeric_col].dropna()) > 1]
    if len(groups) < 2:
        print('Need at least 2 groups with data.')
        return {}
    f_stat, p = stats.f_oneway(*groups)
    result = {'F_statistic': round(f_stat, 4), 'p_value': p,
              'significant_at_05': p < 0.05}
    print(f"ANOVA: F={f_stat:.4f}, p={p:.3e}  ->  "
          f"{'SIGNIFICANT' if p < 0.05 else 'not significant'} at a=0.05")
    return result

anova_test(df, 'call_duration', 'disposition')

---
## Summary

All functions from **EDA_Governance.ipynb** (20 functions) and **EDA_Correlation.ipynb** (12 functions) have been tested against `raw_test_call_data.csv`.